# Cell Line Finder — Transcriptomics + Proteomics Selection

**Input:** a target gene and an exclusion gene, by symbol or ENSG.
**Output:** one table of cell lines with expression and protein calls, and a
plain-language description of the evidence behind every row — including a
stated reason for every missing value.

## The flow

```
resolve both genes
  -> fetch RNA from DepMap / GEO / HPA, DETECT each source's scale
  -> audit for detection floors
  -> DERIVE the scoring thresholds from the data
  -> score each line WITHIN its lineage, per source
  -> combine sources by meta-analysis (Stouffer), check they agree (I2)
  -> GATE on the exclusion gene: pass / fail / unknown
  -> RANK the passing lines on the target
  -> attach protein as VALIDATION
  -> describe every row
```

## The three decisions that shape everything

**Stratify, don't normalise.** Every score is a robust z computed inside a
`(source, lineage)` stratum, which makes it unitless — scale and batch
differences cancel exactly. Verified: identical biology measured by sequencing
and by microarray gives the same z. Quantile-normalising the sources together
would work numerically but would fabricate near-zero values for a source that
floors at background.

**Gate, then rank.** Filtering and ranking are separate steps. A single
conjunctive score would conflate "does this line meet the criteria" with "how
good is it", so a line failing exclusion would still receive a number.

**Flag, never drop.** Small lineages, single-source lines, silent lineages and
unmeasured exclusion genes all stay in the output with the reason attached. A
line removed silently is a line the user cannot evaluate.

## 1. Configuration

Only values that genuinely cannot be estimated are set here. `MAD_FLOOR`,
`Z_STRONG` and `MIN_PEERS_FOR_Z` are **derived per gene** in section 3 and
overwrite these fallbacks.

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import gaussian_kde

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 400)
plt.rcParams.update({"figure.dpi": 110})

DB_PATH = r"C:\Disertation\UoB-GeneTraceAI-25-26\src\pipeline\outputs\celllineselector.db"  # absolute path patch
con = duckdb.connect(DB_PATH, read_only=True)

# ── conventions WITH a citation (deliberately not estimated) ─────────────────
# log2(1 TPM + 1) = 1.0 exactly. 1 TPM is HPA's own published detection
# threshold (Uhlen et al. 2015) — a cited convention, not a tuned knob.
EXPRESSED_MIN = 1.0
I2_CUT        = 50      # I2 > 50% = substantial heterogeneity (Higgins 2002)
HPA_FOLD      = 5.0     # HPA's 5-fold tissue-enriched rule (Uhlen 2015)
DELTA_CUT     = 0.474   # Cliff's delta "large" (Romano et al. 2006)

# ── structural, not statistical ──────────────────────────────────────────────
NO_LINEAGE            = "(no lineage recorded)"  # must not collide with DepMap's "unknown"
MIN_PEERS_FOR_LINEAGE = 15    # below this a lineage is FLAGGED small, never dropped
SILENT_FRAC           = 0.20  # gene off in >80% of a stratum -> z is meaningless
PRIOR_STRENGTH        = 1.0   # shrinkage z*k/(k+prior); stated, not derived
FLOOR_FRAC_LOW        = 0.02  # <2% of values below 1 TPM -> suspect a detection floor

# ── protein ──────────────────────────────────────────────────────────────────
MIN_DETECTED           = 50   # proteins seen in fewer lines are not scored
PROT_MIN_PEERS_LINEAGE = 10
ISOFORM_MIN_R          = 0.3  # below this, isoforms measure different products

# ── DERIVED in section 3; these are only fallbacks ───────────────────────────
MAD_FLOOR       = 0.10
Z_STRONG        = 1.50
PROT_Z_STRONG   = 1.50
MIN_PEERS_FOR_Z = 3
EXCL_Z_CUT      = 1.50
TAU_BROAD, TAU_SPECIFIC = 0.30, 0.60

# Sources allowed to judge SILENCE and TISSUE SPECIFICITY, set empirically by
# source_floor_check(): a floored source reports background as signal, so it can
# neither tell "off" from "below detection" nor support the ratio tau needs.
SILENT_SOURCES = ["depmap_expr", "hpa_rna"]
TAU_SOURCES    = ["depmap_expr", "hpa_rna"]

### Geo Meta Data Download helper Function

In [ ]:
#def harvest_geo_methods(gsms, destdir="./geo_meta", verbose=True):
#    """
#    Fetch per-GSM metadata from NCBI and classify the normalisation method.

#    GEO ARCHIVES; it does not compute. Each GSM carries whatever its submitter
#    uploaded, described in the free-text `data_processing` field and the VALUE
#    column definition — so the method must be read per sample, not assumed for
#    the series.

#    Requires GEOparse (pip install GEOparse). One row per GSM. Cache the result:
#    this makes one network call per accession.
#    """
#    import GEOparse
#    rows = []
#    for i, g in enumerate(gsms, 1):
#        try:
#            s = GEOparse.get_GEO(geo=g.upper(), destdir=destdir, silent=True)
#            m = s.metadata
#            rows.append({
#                "gsm": g.lower(),
#                "platform": m.get("platform_id", [None])[0],
#                "processing": (m.get("data_processing", [""])[0] or "")[:200],
#                "value_def": next((v for k, v in s.columns.description.items()
#                                   if k.upper() == "VALUE"), None),
#                "has_abs_call": "ABS_CALL" in s.table.columns,
#                "series": "; ".join(m.get("series_id", [])),
#                "title": (m.get("title", [""])[0] or "")[:120],
#            })
#        except Exception as e:
#            rows.append({"gsm": g.lower(), "platform": None,
#                         "processing": f"FETCH FAILED: {e}"})
#        if verbose and i % 200 == 0:
#            print(f"  {i:,} / {len(gsms):,}")

#    meta = pd.DataFrame(rows)

    # gcRMA MUST be tested before RMA — the string "gcRMA" contains "RMA"
#    meta["method"] = np.select(
#        [meta.processing.str.contains("gcrma", case=False, na=False),
#         meta.processing.str.contains("rma|robust multi", case=False, na=False),
#         meta.processing.str.contains("mas5|mas 5|gcos", case=False, na=False),
#         meta.processing.str.contains("quantile", case=False, na=False)],
#        ["gcRMA", "RMA", "MAS5", "quantile"], default="other")

#    if verbose:
#        print(f"\n{len(meta):,} GSMs\n")
#        print(meta.platform.value_counts().to_string())
#        print()
#        print(meta.method.value_counts().to_string())
#        print(f"\nwith ABS_CALL (Affymetrix present/absent): {meta.has_abs_call.sum():,}")
#        failed = meta.processing.str.startswith("FETCH FAILED").sum()
#        if failed:
#            print(f"FAILED to fetch: {failed:,}")
#    return meta

#GEO_META_CACHE = "geo_gsm_methods.csv"

#try:
#    meta = pd.read_csv(GEO_META_CACHE)
#    print(f"loaded cached metadata: {len(meta):,} GSMs")
#except FileNotFoundError:
#    gsms = con.execute("SELECT DISTINCT geo_accession FROM geo_info "
#                       "WHERE geo_accession IS NOT NULL").df()["geo_accession"].tolist()
#    meta = harvest_geo_methods(gsms)
#    meta.to_csv(GEO_META_CACHE, index=False)
#    print(f"harvested and cached {len(meta):,} GSMs")

#gsm_method = meta.set_index(meta.gsm.str.lower())["method"]
#print(gsm_method.value_counts().to_string())

## 2. Fetching, with scale detection

Each source arrives on a different scale, and the scale is **detected, not
assumed**:

| source | file / method | scale |
|---|---|---|
| DepMap | `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` | already `log2(TPM+1)` |
| HPA | nTPM — TPM → pTPM → TMM-normalised within source | linear |
| GEO | per-submitter; RMA/gcRMA output log2, MAS5 linear | **varies** |

GEO's `VALUE` column is submitter-defined and need not match the algorithm named
in `data_processing`. This data reports RMA for 72% of samples while storing
**linear** values (RMA subset: min 11.0, median 91.5, max 2,410) — a
metadata-driven rule would have double-logged 2,340 samples, compressing them to
1.5–3.9 without raising an error.

**Why log at all, given everything is z-scored afterwards?** The z is a linear
transform and the log is not, so they do not commute. On raw TPM the same 4-fold
change scores z = +2.83 upward but only −0.71 downward, and the most negative z
reachable is −0.94 — the exclusion gate, which depends on detecting *low*
expression, would barely function. Log makes fold-changes symmetric, which is
what MAD assumes.

**Why nTPM rather than raw TPM for HPA:** nTPM has already been rescaled after
removing non-coding transcripts and TMM-normalised across samples, so cell lines
are comparable to each other. Raw TPM is only comparable *within* a sample,
which is the wrong axis for this method.

In [ ]:
def cols_of(con, t):
    return [r[0] for r in con.execute(f'DESCRIBE "{t}"').fetchall()]


def table_exists(con, t):
    return con.execute("SELECT count(*) FROM information_schema.tables "
                       "WHERE lower(table_name)=?", [t.lower()]).fetchone()[0] > 0


def has_col(con, t, c):
    return table_exists(con, t) and c in cols_of(con, t)


def resolve_gene(con, gene):
    """Accepts an ENSG id or ANY name/alias, in any case."""
    g = str(gene).strip().lower()
    if g.startswith("ensg"):
        q = "SELECT gene_id, gene_names, uniprot_ids, hugo_symbol FROM gene WHERE gene_id = ?"
    else:
        q = ("SELECT gene_id, gene_names, uniprot_ids, hugo_symbol FROM gene "
             "WHERE list_contains(list_transform(gene_names, x -> lower(x)), ?)")
    row = con.execute(q, [g]).fetchone()
    if row is None:
        raise ValueError(f"'{gene}' not found in gene table (as id or name)")
    return row[0], list(row[1] or []), list(row[2] or []), row[3]


def lineage_map(con, table="sample_info", col="lineage"):
    d = con.execute(f'SELECT model_id, "{col}" AS lineage FROM "{table}" '
                    f'WHERE "{col}" IS NOT NULL').df()
    return d.drop_duplicates("model_id").set_index("model_id")["lineage"]


def detect_scale(v, log_max=25.0, log_min=-10.0):
    """
    Is this vector already log2, or linear? The log2 signature is a bounded
    maximum: RMA/gcRMA span roughly 2-14, while MAS5 intensity and TPM run to
    thousands.
    """
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if not len(v):
        return "unknown"
    if v.min() >= log_min and np.nanpercentile(v, 99) < log_max and v.max() <= log_max * 2:
        return "log2"
    return "linear"


def maybe_log(v, name="", force=None, verbose=True):
    """
    Apply log2(x+1) only if the values are not already logged.
    force=True/False overrides the detection; None (default) detects.
    """
    v = np.asarray(v, float)
    scale = detect_scale(v) if force is None else ("linear" if force else "log2")
    do = scale == "linear"
    if verbose:
        print(f"  {name:16s} {scale:7s} (min {np.nanmin(v):8.2f}, max {np.nanmax(v):10.2f})"
              f" -> {'log2(x+1) applied' if do else 'left as-is'}")
    return np.log2(np.clip(v, 0, None) + 1) if do else v


def fetch_gene_expression(con, gene, log_geo=None, log_hpa=None, verbose=True):
    """
    All expression values for one gene, one row per (source, cell line).

    Scale handling, verified against each source's documentation:

      depmap_expr  OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv —
                   ALREADY log2(TPM+1). Never transformed.
      hpa_rna      nTPM: TPM rescaled to pTPM after removing non-coding
                   transcripts, then TMM-normalised within source. Linear.
      geo_expr     depends on the submitter. RMA/gcRMA OUTPUT log2; MAS5
                   outputs linear. But GEO's VALUE column is submitter-defined
                   and this data stores LINEAR values even for RMA samples.

    log_geo / log_hpa default to None = DETECT the scale rather than assume it.
    Passing True forces the transform, False suppresses it. Detection matters
    because applying log2 to already-logged values compresses them to 1.5-3.9
    without erroring — every downstream z would be silently wrong.

    GEO holds several GSM samples per cell line, so it is collapsed to a median
    with n_samples carried forward as a precision weight for Stouffer.
    """
    gid, gnames, _, hugo = resolve_gene(con, gene)
    frames = []
    if verbose:
        print(f"{gene} -> {gid} ({hugo or ', '.join(gnames)})")

    if has_col(con, "depmap_expr", gid):
        d = con.execute(f'SELECT model_id, "{gid}" AS value FROM depmap_expr '
                        f'WHERE "{gid}" IS NOT NULL').df()
        # DepMap ships log2(TPM+1) by filename; checked, never transformed
        if verbose and len(d):
            print(f"  {'depmap_expr':14s} {detect_scale(d['value']):7s} "
                  f"(min {d.value.min():7.2f}, max {d.value.max():9.2f}) -> left as-is")
        d["n_samples"] = 1
        d["source"] = "depmap_expr"
        frames.append(d)

    if has_col(con, "geo_expr", gid):
        d = con.execute(f'SELECT model_id, median("{gid}") AS value, count(*) AS n_samples '
                        f'FROM geo_expr WHERE "{gid}" IS NOT NULL GROUP BY model_id').df()
        if len(d):
            d["value"] = maybe_log(d["value"].to_numpy(), "geo_expr",
                                   force=log_geo, verbose=verbose)
        d["source"] = "geo_expr"
        frames.append(d)

    vcol = next((c for c in ("ntpm", "ptpm", "tpm") if has_col(con, "hpa_rna", c)), None)
    if vcol:
        d = con.execute(f'SELECT model_id, avg("{vcol}") AS value, count(*) AS n_samples '
                        f'FROM hpa_rna WHERE lower(gene)=? GROUP BY model_id', [gid]).df()
        if len(d):
            d["value"] = maybe_log(d["value"].to_numpy(), f"hpa_rna ({vcol})",
                                   force=log_hpa, verbose=verbose)
        d["source"] = "hpa_rna"
        frames.append(d)

    df = pd.concat(frames, ignore_index=True)
    df["lineage"] = df["model_id"].map(lineage_map(con))

    if verbose:
        print()
        print(df.groupby("source").agg(n_lines=("model_id", "nunique"),
              median=("value", "median"), min=("value", "min"),
              max=("value", "max")).round(2).to_string())
        print(f"\nlineages: {df.lineage.nunique()} | "
              f"lines with no lineage: {df.loc[df.lineage.isna(), 'model_id'].nunique()}")
    return df, gid

### 2.1 Detection-floor audit

A source whose minimum sits well above zero **and** which reports essentially no
low values has a detection floor — it is reporting background as signal. RNA-seq
TPM genuinely reaches 0; Affymetrix MAS5/RMA floors at the fitted background (the
observed floor of log2 3.59 is `log2(11+1)`, and MAS5's noise floor is 10–30
intensity units).

This affects **exactly one thing**: judging whether a gene is switched off. The
z-scores of a floored source stay valid, because z is computed within stratum and
the floor cancels.

In [ ]:
def source_floor_check(df, expressed_min=EXPRESSED_MIN, floor_frac=FLOOR_FRAC_LOW):
    """
    Detect a detection floor per source.

    A source whose minimum sits well above zero AND which reports essentially
    no low values has a floor: it is reporting background as signal. Such a
    source cannot judge whether a gene is off, and is excluded from
    SILENT_SOURCES.

    RNA-seq TPM genuinely reaches 0; RMA-normalised microarray floors at the
    fitted background, typically log2 3-6. The signature is a hard minimum with
    zero mass below it.
    """
    rows = []
    for src, g in df.groupby("source"):
        v = g["value"].dropna()
        frac_low = (v < expressed_min).mean()
        rows.append({"source": src, "n": len(v),
                     "min": round(v.min(), 3), "p01": round(v.quantile(.01), 3),
                     "frac_below_1": round(frac_low, 4),
                     "floored": bool(v.min() > expressed_min and frac_low < floor_frac)})
    out = pd.DataFrame(rows)
    floored = out.loc[out.floored, "source"].tolist()
    print(out.to_string(index=False))
    if floored:
        print(f"\nFLOORED: {', '.join(floored)} — excluded from silent-lineage judgements.")
        print("  These sources cannot distinguish 'not expressed' from 'below detection'.")
        print(f"  Their z-scores remain valid: z is computed within source, so the")
        print(f"  floor cancels. Only the silence judgement is affected.")
    else:
        print("\nno floored sources detected — all may vote on silence")
    return out, [s for s in out.source if s not in floored]

## 3. Deriving the thresholds

Three of the four scoring constants are estimable from the data; the fourth is a
published convention and is better left as one.

| constant | derived how | why it matters |
|---|---|---|
| `MAD_FLOOR` | 5th percentile of observed non-zero MADs | a floor below the smallest real spread does nothing; above it, genuine signal is capped |
| `Z_STRONG` | 95th percentile of \|z\| under a within-stratum permutation null | this is the \|z\| reached 5% of the time by chance — which is what "strong" should mean |
| `MIN_PEERS_FOR_Z` | smallest n at which z stops being unstable under resampling | at n=3 the z of a fixed value has **sd ≈ 25** across resamples; unusable |
| `EXPRESSED_MIN` | **not derived** | `log2(1 TPM + 1) = 1.0`, and 1 TPM is HPA's own threshold (Uhlén 2015) |

In [ ]:
def calibrate_constants(df, n_perm=30, seed=0, verbose=True):
    """
    DERIVE the scoring thresholds from the data instead of asserting them.

    Three of the four are genuinely estimable; the fourth is a published
    convention and is better left as one.

      MAD_FLOOR        the 5th percentile of the observed non-zero MADs across
                       strata. A floor below the smallest real spread does
                       nothing; one above it silently caps genuine signal.

      Z_STRONG         the 95th percentile of |z| under a WITHIN-STRATUM
                       permutation null. This is the |z| a line reaches 5% of
                       the time by chance alone, which is what "strong" should
                       mean. A hardcoded 1.5 is often far too permissive.

      MIN_PEERS_FOR_Z  the smallest n at which z stops being unstable, found by
                       resampling. At n=3 the z of a fixed value has sd ~25
                       across resamples, which is unusable.

      EXPRESSED_MIN    NOT derived. log2(1 TPM + 1) = 1.0 exactly, and 1 TPM is
                       HPA's own published detection threshold (Uhlen 2015).
                       A convention with a citation beats an estimate here.
    """
    rng = np.random.default_rng(seed)
    d = df.dropna(subset=["value", "lineage"])

    mads = d.groupby(["source", "lineage"]).value.apply(
        lambda s: stats.median_abs_deviation(s, scale="normal"))
    mads = mads[mads > 0]
    mad_floor = float(np.quantile(mads, 0.05)) if len(mads) else 0.1

    null = []
    for _ in range(n_perm):
        p = d.copy()
        p["value"] = p.groupby(["source", "lineage"]).value.transform(
            lambda s: rng.permutation(s.values))
        for _, sub in p.groupby(["source", "lineage"]):
            v = sub.value.to_numpy()
            m = stats.median_abs_deviation(v, scale="normal")
            if m > 0 and len(v) >= 3:
                null.extend(np.abs((v - np.median(v)) / max(m, mad_floor)))
    z_strong = float(np.quantile(null, 0.95)) if null else 1.5

    grouped = d.groupby(["source", "lineage"]).value.apply(list)
    biggest = max(grouped, key=len) if len(grouped) else []
    min_peers, prev = 3, None
    if len(biggest) >= 60:
        arr = np.asarray(biggest, float)
        for n in [3, 5, 8, 10, 15, 20, 30]:
            if n > len(arr) // 2:
                break
            zs = []
            for _ in range(200):
                sub = rng.choice(arr, n, replace=False)
                m = stats.median_abs_deviation(sub, scale="normal")
                if m > 0:
                    zs.append((arr[0] - np.median(sub)) / max(m, mad_floor))
            sd = float(np.std(zs)) if zs else np.inf
            if prev is not None and sd > 0 and prev / sd < 1.35:
                min_peers = n
                break
            prev, min_peers = sd, n

    out = {"MAD_FLOOR": round(mad_floor, 3), "Z_STRONG": round(z_strong, 2),
           "MIN_PEERS_FOR_Z": int(min_peers), "EXPRESSED_MIN": 1.0}
    if verbose:
        print("derived from this gene's data:")
        for k, v in out.items():
            note = "   (convention: HPA >=1 TPM, Uhlen 2015)" if k == "EXPRESSED_MIN" else ""
            print(f"  {k:18s} {v}{note}")
        print(f"\n  MADs across {len(mads)} strata: min {mads.min():.3f}, "
              f"median {mads.median():.3f}")
        print(f"  null |z| 95th percentile from {n_perm} permutations")
    return out

## 4. Scoring

| step | statistic | handles |
|---|---|---|
| effect size | robust z = (x − median)/MAD **within stratum** | scale and batch; SD would be inflated by the outlier being tested |
| significance | rank-based empirical p | no distributional assumption |
| combination | Stouffer, weighted by √n_samples | grows with agreeing sources, so small consistent deviations become detectable |
| agreement | Cochran's Q → I² | flags sources that disagree |
| shrinkage | z·k/(k+1) | stops unreplicated extremes topping the ranking |

**Why Stouffer rather than averaging.** Averaging discards how many independent
datasets agree. Three sources each at z = 1.5 combine to Z = 2.60 — the same as
one source at z = 2.6 — while three disagreeing sources (2.6, 0.1, −0.3) collapse
to 1.39.

### The three guards

A z divides by spread; when spread collapses the score reports the missing
denominator rather than the biology.

| guard | example | why z fails |
|---|---|---|
| too few peers | `adrenal_cortex`, n=1 | one value has no spread; MAD = 0 by definition |
| gene silent in lineage | EGFR in blood: n=104, median 0.12, 38% at zero | any expression scores z = 16+ |
| no spread | MAD exactly 0 with adequate n | every line reads identically |

**A MAD floor alone does not fix the silent case** — blood's MAD is 0.178, above
any sensible floor, while the distribution is still degenerate. Suppressed lines
are flagged with the reason, never dropped.

In [ ]:
def robust_z(x, v, mad_floor=MAD_FLOOR, min_n=MIN_PEERS_FOR_Z,
             expressed_min=EXPRESSED_MIN, silent_frac=SILENT_FRAC):
    """
    (x - median) / MAD within a lineage. Returns (z, reason).

    z is NaN whenever the reference distribution cannot support one, and
    `reason` names which guard fired. Three guards, each for a real situation
    seen in this data:

      "too few peers"          n < 3. MAD is 0 by construction, not because
                               the lines agree. e.g. adrenal_cortex, n=1.

      "gene silent in lineage" the gene is off across the lineage — median
                               near 0, most lines at zero. A line with any
                               expression then scores z = 16+, which measures
                               the ABSENCE of a denominator, not a real
                               effect. e.g. EGFR in blood: n=104, median 0.12,
                               38% at exactly zero.

                               A MAD floor alone does NOT catch this. Blood's
                               MAD is 0.178 — above any sensible floor — while
                               the distribution is still degenerate. This
                               guard is the one that matters.

      "no spread"              MAD is exactly 0 despite adequate n.

    Lines suppressed here are not dropped; they surface with
    verdict = "insufficient data" and the reason attached, and are reported
    separately in section 5.2.
    """
    v = np.asarray(v, float)
    if len(v) < min_n:
        return np.nan, "too few peers"
    if (v > expressed_min).mean() < silent_frac:
        return np.nan, "gene silent in lineage"
    mad = stats.median_abs_deviation(v, scale="normal")
    if mad <= 0:
        return np.nan, "no spread"
    return (x - np.median(v)) / max(mad, mad_floor), "ok"


def empirical_p(x, v, tail="two"):
    v = np.asarray(v); n = len(v)
    hi = ((v >= x).sum()+1)/(n+1); lo = ((v <= x).sum()+1)/(n+1)
    return min(1.0, 2*min(hi,lo)) if tail=="two" else (hi if tail=="hi" else lo)


def stouffer(zs, weights=None):
    z = np.asarray([v for v in zs if np.isfinite(v)], float)
    if not len(z): return np.nan, np.nan, 0
    w = np.ones(len(z)) if weights is None else np.asarray(
        [w for w,v in zip(weights,zs) if np.isfinite(v)], float)
    Z = (w*z).sum()/np.sqrt((w**2).sum())
    return Z, 2*(1-stats.norm.cdf(abs(Z))), len(z)


def heterogeneity(zs):
    z = np.asarray([v for v in zs if np.isfinite(v)], float); k=len(z)
    if k < 2: return np.nan, np.nan, np.nan
    Q = ((z-z.mean())**2).sum()
    return Q, 1-stats.chi2.cdf(Q,k-1), (max(0,(Q-(k-1))/Q)*100 if Q>0 else 0.0)


def shrink(z, k, prior_strength=1.0):
    return z * k/(k+prior_strength) if np.isfinite(z) else np.nan


def score_gene_lineage(df, min_peers=MIN_PEERS_FOR_LINEAGE, expressed_min=EXPRESSED_MIN,
                       weight_by_samples=True, prior_strength=1.0,
                       replicate_warn=1.0, silent_sources=None, verbose=True):
    """
    Rank every cell line for one gene WITHIN its lineage, combining sources by
    meta-analysis.

    NOTHING is dropped. Lines in small lineages, lines with no lineage label,
    single-source lines, and lines whose lineage does not express the gene are
    all scored where possible and FLAGGED where not.

    silent_sources : sources permitted to judge whether a gene is off. A source
        with a detection floor reports background as signal and cannot tell
        "not expressed" from "below detection". Set this from
        source_floor_check() rather than assuming. Note the z-scores of a
        floored source remain valid — z is computed within source, so the floor
        cancels; only the silence judgement is affected.

    Replicate handling
    ------------------
    A cell line can have several DepMap profiles — separate sequencing runs of
    the same line, each with its own profileid. These are technical replicates,
    not independent evidence:
      * keeping one arbitrarily would discard a real measurement
      * counting them as two sources would inflate the Stouffer combination,
        which assumes sources are INDEPENDENT
    So replicates are collapsed to a median and n_samples is summed — the extra
    profile counts as precision (weight sqrt(2)), not as a second source.
    Replicates differing by more than `replicate_warn` log2 units are reported:
    that is a misidentification signal, not technical variation.

    Returns (summary_df, per_source_df).
    """
    sil_src = silent_sources if silent_sources is not None else SILENT_SOURCES
    d = df.dropna(subset=["value"]).copy()
    # NO_LINEAGE rather than "unknown": DepMap uses "unknown" as a real label,
    # and pooling those with unlabelled lines builds one group from two
    # different situations.
    d["lineage"] = d["lineage"].fillna(NO_LINEAGE)
    if "n_samples" not in d.columns:
        d["n_samples"] = 1

    dup_mask = d.duplicated(["model_id", "source"], keep=False)
    if dup_mask.any():
        dups = d[dup_mask]
        rng_ = dups.groupby(["model_id", "source"])["value"].agg(["min", "max"])
        rng_["range"] = rng_["max"] - rng_["min"]
        if verbose:
            print(f"  {int(dup_mask.sum()):,} rows are replicate profiles of "
                  f"{dups.model_id.nunique():,} cell lines "
                  f"(max within-line range {rng_['range'].max():.2f} log2)")
        suspect = rng_[rng_["range"] > replicate_warn]
        if len(suspect) and verbose:
            print(f"  WARNING: {len(suspect)} line(s) have replicates differing by "
                  f">{replicate_warn} log2 — possible misidentification, not replication")

    d = (d.groupby(["model_id", "source", "lineage"], as_index=False)
           .agg(value=("value", "median"), n_samples=("n_samples", "sum")))

    recs = []
    for (src, lin), g in d.groupby(["source", "lineage"]):
        v = g["value"].to_numpy()
        expressed = v[v > expressed_min]
        small = len(v) < min_peers
        for _, r in g.iterrows():
            z, why = robust_z(r.value, v, expressed_min=expressed_min)
            recs.append({"model_id": r.model_id, "source": src, "lineage": lin,
                         "value": r.value, "n_peers": len(v), "small_lineage": small,
                         "lineage_median": np.median(v), "z": z, "z_reason": why,
                         "pct_expressed": (np.mean(expressed < r.value) * 100
                                           if len(expressed) else np.nan),
                         "p_emp": empirical_p(r.value, v),
                         "n_samples": r.n_samples,
                         "frac_expressed": float((v > expressed_min).mean())})
    per = pd.DataFrame(recs)

    out = []
    for mid, g in per.groupby("model_id"):
        zs = g.z.tolist()
        w = np.sqrt(g.n_samples.astype(float)).tolist() if weight_by_samples else None
        Z, p, k = stouffer(zs, w)
        Q, pq, I2 = heterogeneity(zs)
        ps = g.p_emp.dropna().to_numpy()
        fisher_p = (stats.combine_pvalues(ps, method="fisher")[1] if len(ps) > 1
                    else (ps[0] if len(ps) else np.nan))
        gs = g[g.source.isin(sil_src)]
        out.append({"model_id": mid, "lineage": g.lineage.iloc[0],
                    "z_reasons": "; ".join(sorted(set(g.z_reason))),
                    # Only unfloored sources vote on silence, and .all() rather
                    # than .any(): a lineage can be silent in one source and
                    # expressed in another, and one source must not condemn the
                    # line on its own.
                    "lineage_silent": bool(len(gs) and
                        (gs.frac_expressed < SILENT_FRAC).all()),
                    "lineage_silent_any": bool((g.frac_expressed < SILENT_FRAC).any()),
                    "n_sources": k, "n_peers": int(g.n_peers.max()),
                    "n_profiles": int(g.n_samples.sum()),
                    "small_lineage": bool(g.small_lineage.any()),
                    "mean_value": round(g.value.mean(), 2),
                    "lineage_median": (round(gs.lineage_median.iloc[0], 2)
                                       if len(gs) else np.nan),
                    "z_combined": Z, "z_shrunk": shrink(Z, k, prior_strength),
                    "p_combined": p, "fisher_p": fisher_p, "I2": I2,
                    "sources_agree": (not np.isfinite(I2)) or I2 < I2_CUT,
                    "pct_expressed": g.pct_expressed.mean(),
                    "frac_lineage_expressed": (round(gs.frac_expressed.min(), 3)
                                               if len(gs) else np.nan)})
    res = pd.DataFrame(out)

    ok = res.p_combined.notna()
    res.loc[ok, "q_value"] = stats.false_discovery_control(res.loc[ok, "p_combined"])

    def verdict(r):
        if not np.isfinite(r.z_shrunk):   return f"Insufficient Data ({r.z_reasons})"
        if not r.sources_agree:           return "Conflicting Sources"
        strong = abs(r.z_shrunk) > Z_STRONG   # q reported, not gating (see config)
        d_ = "HIGH" if r.z_shrunk > 0 else "LOW"
        if strong and r.n_sources >= 2 and not r.small_lineage: return d_
        if strong and r.n_sources < 2:    return f"{d_} (Single Source)"
        if strong and r.small_lineage:    return f"{d_} (Small Lineage)"
        if r.z_shrunk >  1.0: return "High (Weak)"
        if r.z_shrunk < -1.0: return "Low (weak)"
        return "Typical"

    res["verdict"] = res.apply(verdict, axis=1)
    for c in ["z_combined", "z_shrunk", "I2", "pct_expressed", "frac_lineage_expressed"]:
        res[c] = res[c].round(2)
    return res.sort_values("z_shrunk", ascending=False).reset_index(drop=True), per

## 5. Which tissue does this gene belong to?

A prior question before ranking. Computed on **unfloored sources only** — a
floored source lifts every silent tissue to background, compressing the ratio τ
depends on and under-calling specificity (observed: GEO τ = 0.666 against DepMap
0.748 and HPA 0.741 on identical biology).

**τ** (Yanai 2005) measures concentration — a gene high in two tissues scores
near 0.5, not 0.9. **HPA's 5-fold rule** is the field convention, run on linear
TPM because the rule is multiplicative. **Cliff's delta** answers *which* tissue,
one-vs-rest.

τ and the z-score are complementary opposites: z is computed *within* lineage and
therefore deletes exactly the between-lineage differences τ measures. Feeding
z-scores to τ would return ~0 for every gene.

In [ ]:
def tau(medians):
    """
    Tissue-specificity index (Yanai et al. 2005, Bioinformatics 21:650).
    0 = uniformly expressed, 1 = expressed in exactly one tissue.
    Measures concentration: a gene high in two tissues scores ~0.5.
    """
    x = np.asarray(medians, float); x = x[np.isfinite(x)]
    if len(x) < 2 or x.max() <= 0:
        return np.nan
    return float(((1 - x/x.max()).sum())/(len(x)-1))


def hpa_classify(medians_linear, detect=1.0, fold=HPA_FOLD):
    """
    HPA's published tissue-specificity scheme (Uhlen et al. 2015).

    Applied on the LINEAR TPM scale — the 5-fold rule is multiplicative, so
    running it on log2 values would silently change the threshold.

      tissue enriched  : one tissue >= fold x every other tissue
      group enriched   : 2-7 tissues, mean >= fold x all others
      tissue enhanced  : >= fold x the mean across tissues
      expressed in all : >= detect everywhere
      not detected     : < detect everywhere
      mixed            : none of the above
    """
    s = pd.Series(medians_linear).dropna().sort_values(ascending=False)
    if len(s) < 2:
        return "insufficient tissues", []
    if (s < detect).all():
        return "not detected", []
    top, rest = s.iloc[0], s.iloc[1:]
    if top >= detect and rest.max() > 0 and top >= fold * rest.max():
        return "tissue enriched", [s.index[0]]
    for k in range(2, min(8, len(s))):
        grp, outg = s.iloc[:k], s.iloc[k:]
        if len(outg) and grp.mean() >= fold * outg.max() and grp.min() >= detect:
            return "group enriched", list(s.index[:k])
    enh = s[s >= fold * s.mean()]
    if len(enh):
        return "tissue enhanced", list(enh.index)
    if (s >= detect).all():
        return "expressed in all", []
    return "mixed", []


def tissue_contrast(df, tissue, value="value", group="lineage", min_n=5):
    """
    One-vs-rest test for a single tissue.

    Cliff's delta = P(in > out) - P(out > in), computed from the Mann-Whitney U.
    Non-parametric, bounded -1..1, and directly interpretable as 'how often does
    a line from this tissue exceed a line from elsewhere'.
    """
    a = df.loc[df[group] == tissue, value].dropna().to_numpy()
    b = df.loc[df[group] != tissue, value].dropna().to_numpy()
    if len(a) < min_n or len(b) < min_n:
        return None
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"tissue": tissue, "n_in": len(a), "n_out": len(b),
            "median_in": round(np.median(a), 2), "median_out": round(np.median(b), 2),
            "cliffs_delta": round(2*U/(len(a)*len(b)) - 1, 3), "p": p}


def tissue_attribution(df, sources=TAU_SOURCES, min_n=5, delta_cut=DELTA_CUT,
                       verbose=True):
    """
    Is this gene tissue-specific, and to which tissue(s)?

    Computed on unfloored sources only. A floored source lifts every silent
    tissue to background, compressing the ratio tau depends on and
    systematically under-calling specificity (observed: GEO tau 0.666 vs
    DepMap 0.748 / HPA 0.741 on identical biology).

    tau and the HPA 5-fold rule are RATIOS, so they run on the linear TPM
    scale — a ratio of log values is not a ratio of expression. Cliff's delta
    is rank-based, so the monotone log is irrelevant to it.

    Lineages with fewer than min_n cell lines are excluded from tau and the
    HPA classification. A "tissue" of one cell line has no meaningful median,
    and without this guard a single line can claim the entire classification:
    epidermoid_carcinoma (n=1) drove tau from 0.63 to 0.88 and was reported as
    the enhanced tissue, while Cliff's delta — which already guards with
    min_n — correctly named the four epithelial lineages.

    Note tau and the z-score are complementary opposites: z is computed WITHIN
    lineage and therefore deletes between-lineage differences, which is
    exactly what tau measures. Feeding z-scores to tau would return ~0 for
    every gene.
    """
    d = df[df.source.isin(sources)].dropna(subset=["lineage", "value"])
    if not len(d):
        print(f"no data from {sources}")
        return None

    # exclude lineages too small to have a median, same floor as tissue_contrast
    n_per_lin = d.groupby("lineage").model_id.nunique()
    keep = n_per_lin[n_per_lin >= min_n].index
    dropped = sorted(set(n_per_lin.index) - set(keep))

    # median of per-source medians -> neither source dominates, and the
    # residual scale difference between them cannot leak into tau
    med_log = (d[d.lineage.isin(keep)]
                 .groupby(["source", "lineage"]).value.median()
                 .groupby("lineage").median())
    med_lin = 2**med_log - 1                       # back to TPM for the ratios

    t = tau(med_lin.values)
    hpa_cls, hpa_tis = hpa_classify(med_lin)

    res = pd.DataFrame([r for r in (tissue_contrast(d, x, min_n=min_n)
                                    for x in med_log.index) if r])
    if len(res):
        res["q"] = stats.false_discovery_control(res.p)
        res = res.sort_values("cliffs_delta", ascending=False).reset_index(drop=True)

    if verbose:
        label = ("UNIFORM" if t < TAU_BROAD else
                 "BROAD" if t < TAU_SPECIFIC else "SPECIFIC")
        print(f"tau = {t:.3f} ({label})   HPA class: {hpa_cls}"
              + (f" -> {', '.join(map(str, hpa_tis))}" if hpa_tis else ""))
        print(f"  sources: {', '.join(sources)}   "
              f"lineages used: {len(keep)} of {len(n_per_lin)}")
        if dropped:
            print(f"  excluded (<{min_n} lines): {', '.join(dropped)}")
        if len(res):
            own = res[(res.q < .05) & (res.cliffs_delta > delta_cut)]
            print(f"  delta > {delta_cut} and q < 0.05: "
                  f"{', '.join(own.tissue) if len(own) else 'none'}")
            print("  " + res.head(8).to_string(index=False).replace("\n", "\n  "))

    return {"tau": t, "hpa_class": hpa_cls, "hpa_tissues": hpa_tis,
            "contrasts": res, "lineage_medians_tpm": med_lin.round(2),
            "lineages_excluded": dropped}

## 6. The exclusion gate

### 6.1 Does a natural threshold exist?

Test before choosing a cut. A data-driven threshold is only more principled than
a stated convention **if the structure it claims to find is really there** — a
unimodal distribution still yields clusters from any clustering algorithm, and
the cut lands wherever the objective puts it: arbitrary *and* unstable across
genes and subsets.

In [ ]:
def distribution_modality(values, bw=None, grid=400, min_prominence=0.15):
    """
    Is this gene's distribution multimodal, i.e. is there a NATURAL low/high
    split, or only one hump?

    Matters because a data-driven threshold is only more principled than a
    convention if the structure it claims to find actually exists. A unimodal
    distribution still yields clusters from any clustering algorithm — the cut
    just lands wherever the objective puts it, which is arbitrary AND unstable
    across genes and subsets.

    Returns (n_modes, valley_position or None, diagnostics).
    """
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    if len(v) < 30: return 1, None, {"n": len(v), "reason": "too few values"}
    kde = stats.gaussian_kde(v, bw_method=bw)
    xs = np.linspace(v.min(), v.max(), grid); d = kde(xs)
    pk = np.where((d[1:-1] > d[:-2]) & (d[1:-1] > d[2:]))[0] + 1
    pk = pk[d[pk] >= min_prominence * d.max()]          # ignore ripples
    if len(pk) < 2:
        return 1, None, {"n": len(v), "peaks": len(pk), "peak_x": xs[pk].round(2).tolist()}
    lo, hi = pk[0], pk[1]
    valley = lo + int(np.argmin(d[lo:hi]))
    return len(pk), float(xs[valley]), {
        "n": len(v), "peaks": len(pk), "peak_x": xs[pk].round(2).tolist(),
        "valley_x": round(float(xs[valley]), 3),
        "valley_depth": round(float(1 - d[valley]/d[pk].min()), 3)}

### 6.2 Three-valued, not boolean

| mode | rule |
|---|---|
| `lineage_low` | z < −cut, relative to lineage peers |
| `not_high` | z < +cut, permissive |
| `not_expressed` | below 1 TPM — absolute, and impossible for ubiquitous genes |
| `data_driven` | below the valley between two modes — only when bimodal |

`pass` / `fail` / **`unknown`**. A line never measured for the exclusion gene is
not a low-exclusion line — it is an unknown, and a boolean filter would silently
drop it. That group may contain the best available candidates.

`not_expressed` returning nothing is a **result**, not a failure: the gene is
expressed everywhere and absolute exclusion is unachievable. The gate says so
rather than quietly substituting a relative low.

In [ ]:
def exclusion_gate(excl_res, mode="lineage_low", expressed_min=EXPRESSED_MIN,
                   z_cut=EXCL_Z_CUT, valley=None, verbose=True):
    """
    Three-valued gate on the exclusion gene: pass / fail / UNKNOWN.

    Three-valued, not boolean, because "measured and fails" and "never
    measured" are different situations and a hard filter would collapse them.
    A line with no exclusion evidence may be the best candidate available; the
    user must see it flagged rather than silently removed.

    modes:
      lineage_low    z < -z_cut. Relative to lineage peers.
      not_high       z < +z_cut. Permissive.
      not_expressed  below the detection threshold. Absolute — but impossible
                     for ubiquitously expressed genes, and the caller is told.
      data_driven    below the valley between two modes of the observed
                     distribution. Only available when the distribution is
                     genuinely bimodal (see distribution_modality).
    """
    z, val = excl_res.z_shrunk, excl_res.mean_value
    if mode == "lineage_low":
        g = np.where(z.isna(), np.nan, (z < -z_cut))
    elif mode == "not_high":
        g = np.where(z.isna(), np.nan, (z < z_cut))
    elif mode == "not_expressed":
        g = np.where(val.isna(), np.nan, (val < expressed_min))
    elif mode == "data_driven":
        if valley is None:
            raise ValueError("data_driven requires a valley; run distribution_modality first")
        g = np.where(val.isna(), np.nan, (val < valley))
    else:
        raise ValueError(f"unknown mode: {mode}")

    out = pd.Series(g, index=excl_res.index).map(
        {1.0: "pass", 0.0: "fail"}).fillna("unknown")
    if verbose:
        print(f"exclusion gate [{mode}]: " +
              ", ".join(f"{k} {v:,}" for k, v in out.value_counts().items()))
        if mode == "not_expressed" and (out == "pass").sum() == 0:
            frac = float((val >= expressed_min).mean())
            print(f"  NO line has this gene below detection — it is expressed in "
                  f"{frac:.0%} of measured lines.\n  Absolute exclusion is not "
                  f"achievable; use 'lineage_low' or 'data_driven'.")
    return out

## 7. Protein — a separate axis

mRNA and protein measure different quantities; cross-line correlation is
typically r ≈ 0.4–0.6 because translation rate and protein half-life intervene.
Folding protein into the Stouffer combination would treat post-transcriptional
regulation as noise and let it cancel real RNA signal.

**Three properties break the RNA machinery**, which is why `robust_z_prot` is
separate:

- values are TMT **log-ratios** against a bridge channel — they centre on 0,
  negative means "below the reference", there is no zero and no detection floor,
  so the silence guard is meaningless
- missingness is **bimodal** (41% of proteins detected everywhere, 28% missing in
  over half the lines) and MNAR cannot be distinguished from MCAR once the
  absolute intensity axis is gone — so nothing is imputed
- **plex is not a stratum**: it was inferred from detection patterns (42 groups of
  9, consistent with TMT10 minus a bridge channel) and permutation testing across
  200 proteins found **0** with a plex effect, so bridge normalisation worked

Isoforms are collapsed by **max**, not median: for target selection the question
is whether *any* form is abundant, and a median across a well-detected and a
poorly-detected isoform reports the detection, not the protein.

In [ ]:
def robust_z_prot(x, v, mad_floor=MAD_FLOOR, min_n=MIN_PEERS_FOR_Z):
    """
    (x - median) / MAD within a lineage. Returns (z, reason).

    Deliberately NOT the RNA robust_z. Proteomics values are TMT log-RATIOS
    against a bridge channel: they centre on 0, negative means "below the
    reference", and there is no zero and no detection floor. The RNA version's
    'gene silent in lineage' guard is therefore meaningless here and is absent.
    Only the too-few-peers and no-spread guards apply.
    """
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < min_n:
        return np.nan, "too few peers"
    mad = stats.median_abs_deviation(v, scale="normal")
    if mad <= 0:
        return np.nan, "no spread"
    return (x - np.median(v)) / max(mad, mad_floor), "ok"


def protein_columns_for_gene(con, gene, prot_cols):
    """UniProt accessions for this gene that actually exist in the proteomics table."""
    g = str(gene).strip().lower()
    row = con.execute(
        "SELECT gene_id, uniprot_ids, hugo_symbol FROM gene "
        "WHERE gene_id = ? OR list_contains(list_transform(gene_names, x -> lower(x)), ?)",
        [g, g]).fetchone()
    if row is None:
        raise ValueError(f"'{gene}' not in gene table")
    have = {c.lower() for c in prot_cols}
    return row[0], [u for u in (row[1] or []) if u.lower() in have], row[2]


def collapse_isoforms(block, how="max", min_r=ISOFORM_MIN_R):
    """
    Several UniProt accessions can map to one gene (390 genes, ~816 columns).

    Default is MAX, not median: for target selection the question is whether ANY
    form of the protein is abundant, and a median across a well-detected and a
    poorly-detected isoform reports the detection rather than the protein.

    Isoforms correlating below min_r are measuring different protein products;
    that is reported so the collapse stays inspectable.
    """
    if block.shape[1] == 1:
        return block.iloc[:, 0], {"n_isoforms": 1, "mean_r": np.nan, "discordant": False}
    c = block.corr().values
    r = np.nanmean(c[np.triu_indices_from(c, 1)])
    v = block.max(axis=1) if how == "max" else block.median(axis=1)
    return v, {"n_isoforms": block.shape[1], "mean_r": round(float(r), 3),
               "discordant": bool(np.isfinite(r) and r < min_r)}


def score_protein(con, gene, prot, how="max", min_detected=MIN_DETECTED,
                  min_peers=PROT_MIN_PEERS_LINEAGE, verbose=True):
    """
    Score every cell line for one gene on the PROTEIN axis.

    Deliberately separate from the RNA meta-analysis. mRNA and protein measure
    different quantities — cross-cell-line correlation is typically r 0.4-0.6,
    because translation rate and protein half-life intervene. Folding protein
    into the Stouffer combination would treat post-transcriptional regulation as
    noise and let it cancel real RNA signal. Two axes, reported side by side.

    Missingness is left as-is, never imputed: it is bimodal (41% of proteins
    detected everywhere, 28% missing in over half the lines), and because the
    log-ratio removed the absolute intensity axis, MNAR cannot be distinguished
    from MCAR. `detected` is carried as its own feature instead.

    Plex is NOT a stratum. It was inferred from detection patterns (42 groups of
    9, consistent with TMT10 minus a bridge channel) and permutation testing
    across 200 proteins found no plex effect at p<0.05, so the bridge-channel
    normalisation was effective.

    Returns (per-line dataframe, metadata dict).
    """
    gid, ucols, hugo = protein_columns_for_gene(con, gene, prot.columns)
    if not ucols:
        if verbose:
            print(f"{gene} -> {gid}: no proteomics columns")
        return None, None

    block = prot[ucols].apply(pd.to_numeric, errors="coerce")
    value, iso = collapse_isoforms(block, how=how)

    lin = (con.execute("SELECT model_id, lineage FROM sample_info").df()
             .drop_duplicates("model_id").set_index("model_id")["lineage"])
    d = pd.DataFrame({"model_id": prot["model_id"], "value": value})
    d["detected"] = d["value"].notna()
    d["lineage"] = d["model_id"].map(lin).fillna("(no lineage recorded)")

    n_det = int(d.detected.sum())
    if verbose:
        print(f"{gene} -> {gid} ({hugo})   uniprot: {', '.join(ucols)}")
        print(f"  isoforms: {iso['n_isoforms']}  mean r: {iso['mean_r']}"
              + ("   DISCORDANT — isoforms measure different products"
                 if iso["discordant"] else ""))
        print(f"  detected in {n_det:,} of {len(d):,} lines ({n_det/len(d):.1%})")
    if n_det < min_detected:
        if verbose:
            print(f"  NOT SCORED: detected in fewer than {min_detected} lines")
        return None, {"gene_id": gid, "hugo": hugo, "n_detected": n_det, **iso}

    obs = d[d.detected]
    rows = []
    for lname, g in obs.groupby("lineage"):
        v = g["value"].to_numpy()
        n_in_lineage = int((d.lineage == lname).sum())
        for _, r in g.iterrows():
            z, why = robust_z_prot(r.value, v)
            rows.append({"model_id": r.model_id, "lineage": lname,
                         "value": round(r.value, 3), "n_peers": len(v),
                         "small_lineage": len(v) < min_peers,
                         "lineage_median": round(float(np.median(v)), 3),
                         "prot_z": z, "z_reason": why,
                         "pct_in_lineage": round((v < r.value).mean()*100, 1),
                         "lineage_detect_rate": round(len(g)/max(n_in_lineage, 1), 2)})
    res = pd.DataFrame(rows)

    ok = res.prot_z.notna()
    if ok.any():
        res.loc[ok, "p_emp"] = 2*(1 - stats.norm.cdf(res.loc[ok, "prot_z"].abs()))
        res.loc[ok, "q_value"] = stats.false_discovery_control(res.loc[ok, "p_emp"])

    def verdict(r):
        if not np.isfinite(r.prot_z):
            return f"insufficient data ({r.z_reason})"
        strong = (r.get("q_value", 1) < .05) and abs(r.prot_z) > 1.5
        d_ = "HIGH" if r.prot_z > 0 else "LOW"
        if strong and r.small_lineage: return f"{d_} protein (small lineage)"
        if strong:                     return f"{d_} protein"
        if r.prot_z >  1.0: return "high protein (weak)"
        if r.prot_z < -1.0: return "low protein (weak)"
        return "typical"

    res["verdict"] = res.apply(verdict, axis=1)
    res["prot_z"] = res.prot_z.round(2)

    meta = {"gene_id": gid, "hugo": hugo, "uniprot": ucols, "n_detected": n_det,
            "detect_rate": round(n_det/len(d), 3), **iso}
    if verbose:
        print(f"  scored {int(ok.sum()):,} lines")
        print("  " + res.verdict.value_counts().to_string().replace("\n", "\n  "))
    return res.sort_values("prot_z", ascending=False).reset_index(drop=True), meta

## 8. Selection — annotate, never filter

Every cell line the **target** gene was measured in appears in the output. The
exclusion gene, when supplied, **annotates** rather than removes.

**Why nothing is filtered.** The exclusion criterion is a judgement about the
user's experiment, not a fact about the cell line. A line that fails is still
worth seeing — labelled — because the user may want to reconsider the
criterion, or may find the failure informative. Hiding rows makes the tool
harder to trust than labelling them.

**The exclusion gene is optional.** With `exclusion_gene=None` this is the
single-gene case: rank every line on the target alone, and the `exclusion`
column reads `n/a`.

**Missingness stays asymmetric in the labelling**, even though nothing is
dropped:

| missing | label |
|---|---|
| target evidence | tier 5 — cannot be ranked, but still listed |
| exclusion evidence | `unknown`, never `pass` — recommending it would hand the user a line that may strongly express the gene they said to avoid |

**Tiers order the table; they do not gate it.**

| tier | meaning |
|---|---|
| 1 | target HIGH, ≥2 agreeing sources, adequate lineage, exclusion passes |
| 2 | supported, but weaker target or thinner evidence |
| 3 | exclusion never measured, or protein contradicts it |
| 4 | fails the exclusion criterion |
| 5 | target not measurable in this line |

**Protein is validation, not an entry requirement** — it covers ~375 of ~1,580
lines, so demanding it up front would discard candidates for panel membership
rather than biology. `require_target_protein=True` gives the strict view for
drug-target work, where the compound binds the protein and mRNA is a proxy.

In [ ]:

def select_cell_lines(con, target_gene, exclusion_gene=None,
                      exclusion_mode="lineage_low", prot=None,
                      require_target_protein=False, expressed_min=1.0,
                      z_cut=1.5, z_strong=1.5, min_peers_lineage=15,
                      fetch=None, score=None, floor=None, score_prot=None,
                      verbose=True):
    """
    Score every cell line for a TARGET gene, optionally annotating each with an
    EXCLUSION gene. NOTHING is filtered out — every cell line the target gene
    was measured in appears in the output, annotated with why it is or is not
    a good candidate.

    exclusion_gene=None runs the single-gene case: rank on the target alone.

    When an exclusion gene IS supplied it ANNOTATES rather than filters. A line
    that fails the exclusion criterion still appears, marked `fail`, because
    the user may want to see it.

    Missingness stays asymmetric in the LABELLING:
      missing target evidence     -> cannot be ranked; shown, tier 4
      missing exclusion evidence  -> shown as `unknown`, never as `pass`

    Protein is validation, not an entry requirement.
    """

    # Treat string/empty representations of None as actual None
    if exclusion_gene is None:
        exclusion_gene = None
    elif isinstance(exclusion_gene, str) and exclusion_gene.strip().lower() in {
        "", "none", "null", "nan"
    }:
        exclusion_gene = None

    fetch = fetch or fetch_gene_expression
    score = score or score_gene_lineage
    floor = floor or source_floor_check

    def _score(gene):
        e, gid = fetch(con, gene, verbose=False)
        try:
            _, sil = floor(e, verbose=False)
        except TypeError:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                out = floor(e)
            sil = out[1] if isinstance(out, (tuple, list)) and len(out) > 1 else SILENT_SOURCES
        r, _ = score(e, silent_sources=sil, verbose=False)
        return r.set_index("model_id"), gid

    t, gid_t = _score(target_gene)
    if verbose:
        print(f"target    {target_gene:>12s} -> {gid_t}: {len(t):,} lines scored")

    x, gid_x, valley = None, None, None
    if exclusion_gene is not None:
        x, gid_x = _score(exclusion_gene)
        if verbose:
            print(f"exclusion {exclusion_gene:>12s} -> {gid_x}: {len(x):,} lines scored")

    # ---- base is EVERY line the target was measured in; nothing dropped ----
    idx = t.index
    m = pd.DataFrame({
        "model_id": idx,
        "lineage": t["lineage"].values,
        "rna_z_t": t["z_shrunk"].values,
        "rna_val_t": t["mean_value"].values,
        "n_src_t": t["n_sources"].values,
        "n_peers_t": t["n_peers"].values,
        "agree_t": t["sources_agree"].values,
        "i2_t": t["I2"].values if "I2" in t else np.nan,
        "small_lin_t": t["small_lineage"].values,
        "z_reason_t": t["z_reasons"].values if "z_reasons" in t else "",
    })

    if x is not None:
        for src, dst in [("z_shrunk", "rna_z_x"), ("mean_value", "rna_val_x"),
                         ("n_sources", "n_src_x"), ("z_reasons", "z_reason_x")]:
            m[dst] = m.model_id.map(x[src]) if src in x else np.nan
        n_modes, valley, diag = distribution_modality(m["rna_val_x"].dropna())
        if verbose:
            print(f"\nexclusion distribution: {n_modes} mode(s)  {diag}")
            print("  UNIMODAL — no natural low/high split; the chosen mode is a "
                  "stated convention." if n_modes == 1 else
                  f"  BIMODAL — natural split at {valley:.2f}; "
                  f"exclusion_mode='data_driven' is available.")
        gate = exclusion_gate(
            pd.DataFrame({"z_shrunk": m["rna_z_x"], "mean_value": m["rna_val_x"]}),
            mode=exclusion_mode, expressed_min=expressed_min, z_cut=z_cut,
            valley=valley, verbose=verbose)
        m["exclusion"] = gate.values
    else:
        m["exclusion"] = "n/a"
        for c in ["rna_z_x", "rna_val_x", "n_src_x", "z_reason_x"]:
            m[c] = np.nan

    # ---- protein as validation ----
    for gene, suf in [(target_gene, "t"), (exclusion_gene, "x")]:
        m[f"prot_z_{suf}"] = np.nan
        if prot is None or gene is None:
            continue
        sp = score_prot or score_protein
        pr, _ = sp(con, gene, prot, verbose=False)
        if pr is not None:
            m[f"prot_z_{suf}"] = m.model_id.map(pr.set_index("model_id")["prot_z"]).values

    def status(r):
        if not np.isfinite(r.prot_z_t):                     return "not measured"
        if not np.isfinite(r.rna_z_t):                      return "protein only"
        if r.rna_z_t >  z_strong and r.prot_z_t >  z_strong: return "confirms (both high)"
        if r.rna_z_t < -z_strong and r.prot_z_t < -z_strong: return "confirms (both low)"
        if r.rna_z_t >  z_strong and r.prot_z_t < -1.0:      return "CONTRADICTS"
        return "measured, neutral"
    m["protein_status"] = m.apply(status, axis=1)
    # protein can only CONTRADICT exclusion, never confirm it: absence from a
    # TMT run is ambiguous, whereas DepMap and HPA genuinely reach zero
    m["excl_protein_warning"] = m.prot_z_x > z_strong

    def tier(r):
        if not np.isfinite(r.rna_z_t):                      return 5  # cannot rank
        if r.exclusion == "fail":                           return 4  # criterion not met
        if r.excl_protein_warning is True:                  return 3  # protein contradicts
        if r.exclusion == "unknown":                        return 3  # never measured
        strong = (r.n_src_t >= 2 and bool(r.agree_t) and not bool(r.small_lin_t)
                  and (r.exclusion in ("pass", "n/a")))
        if not strong:                                      return 2
        return 1 if r.rna_z_t > z_strong else 2
    m["tier"] = m.apply(tier, axis=1)
    m["tier_label"] = m.tier.map({
        1: "1 - target HIGH, well supported" + ("" if exclusion_gene is None
                                                else ", exclusion passes"),
        2: "2 - supported, weaker target or thinner evidence",
        3: "3 - exclusion UNVERIFIED or protein warning",
        4: "4 - fails the exclusion criterion",
        5: "5 - target not measurable in this line"})

    if require_target_protein:
        m = m[m.prot_z_t.notna()]

    m = m.sort_values(["tier", "rna_z_t"], ascending=[True, False]).reset_index(drop=True)
    if verbose:
        print(f"\n{len(m):,} cell lines returned — none filtered out")
        print(m.tier_label.value_counts().sort_index().to_string())
        top = m[m.tier <= 2]
        if len(top):
            print(f"\ntier 1-2 by lineage (top 8):")
            print("  " + top.lineage.value_counts().head(8).to_string().replace("\n", "\n  "))
    return m

## 9. The report

One row per cell line with a plain-language description: what was measured, where
sources agreed or disagreed, and — critically — **why any value is missing**.
Every NaN carries a stated reason rather than an empty cell.

In [ ]:
def _lvl(z, hi, lo=None):
    """Map a z to a plain-language level. Returns 'unknown' for NaN."""
    lo = lo if lo is not None else -hi
    if not np.isfinite(z): return "unknown"
    if z >= hi:      return "HIGH"
    if z <= lo:      return "LOW"
    if z >= hi*0.66: return "moderately high"
    if z <= lo*0.66: return "moderately low"
    return "typical"


def describe_line(r, target, exclusion, z_strong, prot_strong):
    """
    Plain-language account of ONE cell line: what was measured, what the numbers
    mean, where the sources disagreed, and — critically — WHY any value is
    missing. Every NaN gets a stated reason rather than an empty cell.

    Reads columns defensively with .get(), so it works whether it is handed the
    output of select_cell_lines (rna_z_t / n_src_t / ...) or a raw score frame.
    """
    g = (lambda k, d=np.nan: r[k] if k in r and pd.notna(r[k]) else d)
    lineage = g("lineage", "unknown lineage")
    p = []

    # ---------- target RNA ----------
    zt = g("rna_z_t")
    if np.isfinite(zt):
        npeers = g("n_peers_t", g("n_peers", None))
        nsrc   = g("n_src_t", g("n_sources", None))
        bits = f"{target} RNA is {_lvl(zt, z_strong).lower()} for a {lineage} line (z={zt:+.2f}"
        if npeers is not None and np.isfinite(npeers): bits += f" vs {int(npeers)} lineage peers"
        if nsrc is not None and np.isfinite(nsrc):
            bits += f", {int(nsrc)} source{'s' if int(nsrc) != 1 else ''}"
        p.append(bits + ")")
        if nsrc is not None and np.isfinite(nsrc) and int(nsrc) == 1:
            p.append("only ONE source measured it, so the estimate is unreplicated "
                     "and its score was shrunk by half")
        elif g("agree_t", True) in (False, 0):
            i2 = g("i2_t", g("I2", np.nan))
            p.append(f"the sources DISAGREE"
                     + (f" (I2={i2:.0f}%)" if np.isfinite(i2) else "")
                     + ", so the combined value is unreliable")
        else:
            i2 = g("i2_t", g("I2", np.nan))
            p.append("sources agree" + (f" (I2={i2:.0f}%)" if np.isfinite(i2) else ""))
    else:
        p.append(f"{target} RNA has NO z-score — "
                 f"{g('z_reason_t', 'not measured in this line')}")

    # ---------- target protein ----------
    pt = g("prot_z_t")
    if np.isfinite(pt):
        p.append(f"{target} protein is {_lvl(pt, prot_strong).lower()} (z={pt:+.2f})")
        st = str(g("protein_status", ""))
        if st.startswith("confirms"):
            p.append("protein CONFIRMS the RNA call — two independent axes agree")
        elif st.startswith("CONTRADICTS"):
            p.append("protein CONTRADICTS the RNA call, suggesting post-transcriptional "
                     "control: the transcript is present but the protein is not")
    else:
        p.append(f"{target} protein was not measured here "
                 f"(the proteomics panel covers ~375 of ~1,580 lines)")

    # ---------- exclusion (skipped entirely when no exclusion gene) ----------
    if exclusion is None:
        if g("small_lin_t", False) in (True, 1):
            npeers = g("n_peers_t", np.nan)
            p.append(f"the {lineage} lineage has only "
                     + (f"{int(npeers)} " if np.isfinite(npeers) else "a few ")
                     + "measured lines, so the reference distribution is thin")
        return ". ".join(p) + "."

    zx = g("rna_z_x")
    excl = str(g("exclusion", "unknown"))
    if np.isfinite(zx):
        verdict = {"pass": "PASSES", "fail": "FAILS"}.get(excl, "is UNRESOLVED")
        p.append(f"{exclusion} RNA is {_lvl(zx, z_strong).lower()} (z={zx:+.2f}), "
                 f"so the exclusion criterion {verdict}")
    else:
        p.append(f"{exclusion} RNA has NO z-score — "
                 f"{g('z_reason_x', 'not measured in this line')}; the exclusion "
                 f"criterion cannot be verified")
    if g("excl_protein_warning", False) in (True, 1):
        px = g("prot_z_x")
        p.append(f"WARNING: {exclusion} PROTEIN is high"
                 + (f" (z={px:+.2f})" if np.isfinite(px) else "")
                 + " despite the RNA passing — protein presence is hard evidence "
                   "the gene is there")

    # ---------- caveats ----------
    if g("small_lin_t", False) in (True, 1):
        npeers = g("n_peers_t", np.nan)
        p.append(f"the {lineage} lineage has only "
                 + (f"{int(npeers)} " if np.isfinite(npeers) else "a few ")
                 + "measured lines, so the reference distribution is thin")
    return ". ".join(p) + "."


def build_report(sel, target, exclusion, z_strong=1.5, prot_strong=1.5, top=None):
    """
    The single deliverable: one row per cell line, with a plain-language
    description covering agreements, disagreements, and the reason behind every
    missing value.
    """
    d = sel.copy()
    zt = d["rna_z_t"] if "rna_z_t" in d else pd.Series(np.nan, index=d.index)
    pt = d["prot_z_t"] if "prot_z_t" in d else pd.Series(np.nan, index=d.index)
    d["expression_call"] = [_lvl(z, z_strong) for z in zt]
    d["protein_call"]    = [_lvl(z, prot_strong) for z in pt]
    d["description"] = d.apply(
        lambda r: describe_line(r, target, exclusion, z_strong, prot_strong), axis=1)
    cols = ["model_id", "lineage", "tier_label",
            "rna_z_t", "expression_call", "prot_z_t", "protein_call"]
    if exclusion is not None:
        cols += ["rna_z_x", "exclusion", "prot_z_x"]
    cols += ["description"]
    out = d[[c for c in cols if c in d.columns]]
    return out.head(top) if top else out

## 10. Run

Set `EXCLUSION_GENE = None` for the single-gene case, or name a gene to have
every line annotated against it. Either way **all cell lines are returned** —
the exclusion criterion labels rows, it does not remove them.

In [ ]:
TARGET_GENE    = "DLG1"
EXCLUSION_GENE = "ADAM17"  # or None for a single-gene query

expr_t, gid_t = fetch_gene_expression(con, TARGET_GENE)

### 10.1 Audit the sources, then derive the thresholds

`source_floor_check` decides which sources may judge silence;
`calibrate_constants` replaces the fallback thresholds with values estimated
from this gene's own distributions.

In [ ]:
floor_tbl, silent_sources = source_floor_check(expr_t)
print()
K = calibrate_constants(expr_t)

MAD_FLOOR       = K["MAD_FLOOR"]
Z_STRONG        = K["Z_STRONG"]
MIN_PEERS_FOR_Z = K["MIN_PEERS_FOR_Z"]
EXCL_Z_CUT      = Z_STRONG
PROT_Z_STRONG   = Z_STRONG

### 10.2 Which tissue does the target belong to?

In [ ]:
attrib = tissue_attribution(expr_t)

### 10.3 Score every cell line

Nothing is filtered. `tier` orders the table; `exclusion` labels each row
`pass` / `fail` / `unknown`, or `n/a` when no exclusion gene was given.

In [ ]:
prot = None  # protein layer scored separately in 02_core_score.ipynb


selected = select_cell_lines(con, TARGET_GENE, EXCLUSION_GENE,
                             exclusion_mode="lineage_low", prot=prot,
                             z_cut=EXCL_Z_CUT, z_strong=Z_STRONG)

report = build_report(selected, TARGET_GENE, EXCLUSION_GENE,
                     z_strong=Z_STRONG, prot_strong=PROT_Z_STRONG)
print(f"{len(report):,} cell lines")

### 10.4 The deliverable

One table, every cell line. `description` explains each call, every agreement
and disagreement, and the reason behind every missing value.

In [ ]:
report = build_report(selected, TARGET_GENE, EXCLUSION_GENE,
                     z_strong=Z_STRONG, prot_strong=PROT_Z_STRONG)
print(f"{len(report):,} cell lines\n")
report.head(100)

### 10.5 Read the descriptions

In [ ]:
for _, r in report.head(5).iterrows():
    print(f"{r.model_id}  ({r.lineage})   expression {r.expression_call} | "
          f"protein {r.protein_call}")
    print(f"   {r.description}\n")

### 10.6 The rows that need a second look

Nothing was removed, so these are all still in `report` — this just surfaces
them. Tier 3 lines have an unverified exclusion or a protein warning; tier 4
fail the criterion outright.

In [ ]:
for t in [3, 4, 5]:
    sub = report[report.tier_label.str.startswith(str(t))]
    if not len(sub):
        continue
    print(f"tier {t}: {len(sub):,} lines — {sub.tier_label.iloc[0][4:]}")
    print(f"   e.g. {sub.iloc[0].model_id}: {sub.iloc[0].description[:180]}...\n")

### 10.7 Single-gene query

The same call with `exclusion_gene=None`. The exclusion columns disappear from
the report rather than being filled with NaN.

In [ ]:
single = select_cell_lines(con, TARGET_GENE, None, prot=prot,
                           z_strong=Z_STRONG, verbose=False)
single_report = build_report(single, TARGET_GENE, None, z_strong=Z_STRONG,
                             prot_strong=PROT_Z_STRONG)
print(f"{len(single_report):,} cell lines | columns: {list(single_report.columns)}\n")
print(single_report.iloc[0].description)

### 10.8 How much does the exclusion mode change the labelling?

It no longer changes *which* lines are returned — only how they are labelled.
Still worth knowing: on real data this parameter moves the pass/fail split more
than any statistical choice in the pipeline.

In [ ]:
for mode in ["not_high", "lineage_low", "not_expressed"]:
    s_ = select_cell_lines(con, TARGET_GENE, EXCLUSION_GENE,
                           exclusion_mode=mode, prot=prot,
                           z_cut=EXCL_Z_CUT, z_strong=Z_STRONG, verbose=False)
    vc = s_.exclusion.value_counts()
    print(f"{mode:16s} pass {vc.get('pass', 0):>5,}   fail {vc.get('fail', 0):>5,}   "
          f"unknown {vc.get('unknown', 0):>5,}   tier1 {int((s_.tier == 1).sum()):>4}")

### 10.9 Export

In [ ]:
#suffix = f"_not_{EXCLUSION_GENE}" if EXCLUSION_GENE else ""
#fn = f"cellline_finder_{TARGET_GENE}{suffix}.csv"
#report.to_csv(fn, index=False)
#print(f"wrote {len(report):,} rows to {fn}")

## 11. Validation

Three tests, each able to **fail** — a validator that always passes proves
nothing. Calibration permutes within stratum so all real signal is destroyed but
structure survives; recovery plants deviations of known size and measures what
fraction is found.

In [ ]:
def validate_null_calibration(df, n_perm=200, seed=0, **kw):
    """
    Under the null there is no real signal, so a calibrated method should call
    ~5% of lines significant at q<0.05 and no more. Values are permuted WITHIN
    each (source, lineage) so the lineage structure is preserved and only the
    line-to-value assignment is destroyed.
    """
    rng = np.random.default_rng(seed)
    rates = []
    for _ in range(n_perm):
        d = df.copy()
        d["value"] = (d.groupby(["source","lineage"], dropna=False)["value"]
                        .transform(lambda s: rng.permutation(s.values)))
        r,_ = score_gene_lineage(d, verbose=False, **kw)
        q = r.q_value.dropna()
        rates.append((q < .05).mean() if len(q) else 0.0)
    rates = np.array(rates)
    return {"mean_false_positive_rate": rates.mean(),
            "p95": np.percentile(rates,95), "max": rates.max(),
            "calibrated": rates.mean() <= 0.07}


def validate_recovery(df, n_planted=20, effect_sizes=(0.5,1.0,2.0,3.0),
                      lineage="lung", seed=0, **kw):
    """
    Plant known deviations of known size and check the method finds them.
    Reports sensitivity per effect size — this is what tells you how SMALL a
    deviation the method can actually detect.
    """
    rng = np.random.default_rng(seed)
    base = df[df.lineage==lineage]
    sd = base.groupby("source")["value"].apply(
        lambda s: stats.median_abs_deviation(s, scale="normal")).mean()
    med = base.groupby("source")["value"].median().mean()

    rows=[]
    for eff in effect_sizes:
        d = df.copy()
        planted=[]
        for i in range(n_planted):
            mid=f"PLANT_{eff}_{i}"; planted.append(mid)
            for src in df.source.unique():
                rows_ = {"model_id":mid,"source":src,
                         "value":med + eff*sd + rng.normal(0,.15),
                         "lineage":lineage,"n_samples":1}
                d = pd.concat([d, pd.DataFrame([rows_])], ignore_index=True)
        r,_ = score_gene_lineage(d, verbose=False, **kw)
        got = r[r.model_id.isin(planted)]
        rows.append({"effect_size_mad":eff,"n_planted":n_planted,
            "detected_q05":int((got.q_value<.05).sum()),
            "sensitivity":round((got.q_value<.05).mean(),3),
            "median_z_shrunk":round(got.z_shrunk.median(),2)})
    return pd.DataFrame(rows)


def validate_source_count(df, seed=0, **kw):
    """
    Does the method correctly prefer replicated evidence? Takes real lines with
    3 sources, and re-scores them after hiding 2 sources. A well-behaved method
    should shrink the single-source version harder.
    """
    rng=np.random.default_rng(seed)
    full,_ = score_gene_lineage(df, verbose=False, **kw)
    three = full[full.n_sources==3].model_id.head(30).tolist()
    d = df[~((df.model_id.isin(three)) & (df.source!="depmap_expr"))]
    one,_ = score_gene_lineage(d, verbose=False, **kw)
    m = (full[full.model_id.isin(three)][["model_id","z_combined","z_shrunk","q_value"]]
         .merge(one[one.model_id.isin(three)][["model_id","z_combined","z_shrunk","q_value"]],
                on="model_id", suffixes=("_3src","_1src")))
    m["z_shrunk_drop"] = (m.z_shrunk_3src - m.z_shrunk_1src).round(2)
    return m

In [ ]:
null_res = validate_null_calibration(expr_t, n_perm=30, silent_sources=silent_sources)
for k, v in null_res.items():
    print(f"  {k:26s} {v}")
print()
print(validate_recovery(expr_t, n_planted=20,
                        silent_sources=silent_sources).to_string(index=False))

## 12. Limitations

**Sensitivity is measured, not assumed.** The recovery curve is the honest limit —
deviations below ~2 MAD are visible in `z_shrunk` but do not survive correction,
and should be described as suggestive.

**Percentiles are within this cell line panel**, not across human tissue. High
here means high relative to cancer cell lines — the right comparison for cell
line selection, but not a statement about normal biology.

**GEO has a detection floor**, so it is excluded from silence and τ judgements.
Its z-scores remain valid because z is within-stratum.

**DepMap and HPA are not fully independent.** Both are RNA-seq TPM-derived,
differing mainly in aligner (RSEM vs Kallisto) and normalisation. Stouffer
assumes independence, so agreement between them is weaker evidence than
agreement between different assay types would be.

**The exclusion mode dominates the result** — more than any statistical choice
here. It is a domain decision and should be made deliberately, not defaulted.

**Shrinkage strength is a judgement**, not a derivation. `PRIOR_STRENGTH = 1.0`
means one source retains half its z and three retain three quarters.

**Lineage labels are curated metadata.** A line clustering away from its label in
expression space is usually a misidentification signal, and belongs in a separate
QC analysis rather than a regrouping here.

In [ ]:
con.close()
print("connection closed")